### `Tool Calling`
- 실시간 정보나 정밀하고 복잡한 작업을 하기 힘든 GPT의 한계를 극복하기 위한 기법
- GPT에게 특정 도구와 그에 대한 설명서를 제공하고, 상황에 따라 해당 도구를 호출하여 작업을 진행하도록 설정하는 방법
- 도구는 사용자 정의 함수로 `단순 코드 로직, 웹 검색, DB 연결` 등을 설정할 수 있음

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime, timedelta

In [2]:
load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
client = OpenAI(api_key=MY_API_KEY)

## 1. 도구 사용 없이 현재 시간 질문

In [4]:
completion = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role':'user', 'content':'지금 시간이 몇시야?'}]
)

print(completion.choices[0].message.content)
# gpt-4o라고 해도 현재 시간을 바로 확인할 방법이 없음

죄송하지만, 현재 시간을 알려드릴 수는 없습니다. 기기의 시계를 확인해 주시기 바랍니다.


In [ ]:
completion.choices[0].message   # tool_calls=None

ChatCompletionMessage(content='죄송하지만, 현재 시간을 알려드릴 수는 없습니다. 기기의 시계를 확인해 주시기 바랍니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

## 2. Tool Calling을 활용하여 현재 시간을 응답하는 AI 구현

### 1) 함수(도구) 선언 및 세부사항 설정

In [5]:
# AI 응답 함수 (해당 함수는 도구는 아님!)
def get_ai_response(messages, tools=None) :
    completion = client.chat.completions.create(
        model='gpt-4o',     # 모델(두뇌) 설정
        messages=messages,
        tools=tools         # 도구(함수) 설정
    )
    
    return completion

In [5]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2026-04-01 13:54:41'

In [4]:
# 현재 시간을 가져오는 도구
def get_current_time() :
    # now().strftime : string format time. 현재 시간을 문자열로 가져오기
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return current_time

#### JSON 형태의 도구 명세서(Tool Calling)

In [ ]:
# AI가 사용할 도구 사용 명세서 정의 (JSON형식)
tools = [
    {
        "type": "function",
            # <type 종류>
            # function: 사용자가 정의한 함수를 호출하여 사용하는 것으로 가장 많이 활용됨
            #           (코드로직, API호출, DB쿼리, 사내시스템 연동 등)
            #           ※ 주의: 함수의 return 자료형이 무엇이든 결국 LLM에겐 문자열로 넣어야함
            # code_interpreter: AI가 직접 Python 코드 작성 및 실행
            #                   (데이터 분석, 시각자료 생성, 복잡한 수학계산 등)
            # file_search: 업로드된 문서에서 관련 정보 검색(RAG)
            #              (사내 규정집, 제품 메뉴얼 등 대규모 문서 기반 답변 생성)
            #              ※ 단, openai 서버에 파일 데이터가 다 올라가서 보안 리스크가 있음
        
        "function": {
            # 도구명(함수명)
            "name": "get_current_time",

            # 도구에 대한 설명 (LLM이 도구를 선택할 때 판단하는 근거~~~!)
            # (단순 명사보다는 문장으로 상세히 써주는게 일반적으로 더 성능이 좋음)
            "description": "현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS'형태의 문자열(string)로 반환합니다.",

            # 해당 도구(함수)를 실행시 파라미터(매개변수) 컨트롤
            # (파라미터가 없는 함수는 '없다'고 선언해주기!! 할루시네이션 방지에 좋음)
            # ※ parameters 항목을 설정하지 않으면 모델이 임의로 넣어서 동작시키기도 함
            "parameters": {
                "type": "object",  # type은 object(딕셔너리 형태)로 고정
                "properties": {}   # 파라미터명 (없으면 비워두기)
            }
        }
    }
]

### 2) 멀티턴 대화 AI 모델 구동

In [ ]:
# 초기 시스템 프롬프트 설정
messages = [
    {"role": "system", "content": "너는 사용자의 질문에 맞게 친절히 응답하는 응대원이야."}
]

print("대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.")

while True :
    user_input = input("사용자\t:")

    if user_input == "종료" :
        print("=== 대화를 종료 합니다 ===")
        break

    # 대화 기록(messages)에 사용자 메시지 추가
    messages.append({"role":"user", "content":user_input})

    # 모델 응답 요청 (도구 명세서 설정)
    ai_response = get_ai_response(messages, tools=tools)

    # 반환값 확인용 코드
    print(ai_response.choices[0].message)

    ai_message = ai_response.choices[0].message

    # 모델의 응답에서 tool_calls(도구 사용)값 가져오기(tool_calls의 자료형은 리스트)
    tool_calls = ai_message.tool_calls

    # 도구 사용이 있는 경우(디폴트는 None)
    if tool_calls :
        # 모델이 도구를 쓰겠다고 한 내역 메시지를 먼저 대화 기록에 저장
        # (사용자의 질의에 대해 어떤 도구를 사용할지 결정한 내용을 메시지로 기록해둬야만
        #  아래에 작성하게 될 도구 실행 결과 코드가 동작함 <- OpenAI API 규칙)
        messages.append(ai_message)

        tool_name = tool_calls[0].function.name    # 사용한 도구명(함수명)
        tool_call_id = tool_calls[0].id            # 도구 id값
        # 여기서 [0]을 써서 무조건 '첫 번째' 상자만 열어봤음! 그래서 2개 요청하면 에러 뜬 거임

        # 사용한 도구명이 현재 시간 함수(get_current_time) 라면
        if tool_name == "get_current_time" :
            # 도구를 사용할 때는 철저한 순서와 규격을 지켜 messages에 대화 기록을
            # 남겨야만 모델이 오류 없이 최종 답변을 생성할 수 있음
            messages.append({
                # tool: 메시지 역할 중 도구를 사용한 내역이라는 의미
                # (위에 모델 도구 사용 메시지(append(ai_message))가 저장돼 있어야 함)
                "role": "tool",
                # 도구 id
                "tool_call_id": tool_call_id,
                # 도구명
                "name": tool_name,
                # content는 도구를 사용한 결과물로 반드시 문자열이어야 함
                # (이후 LLM이 최종 답변을 하는데 같이 반영될 것이기 때문)
                "content": get_current_time()  # 현재 함수 반환값은 문자열!   
            })

        ai_response = get_ai_response(messages, tools=tools)
        # 메시지 객체 내 tool_calls 정보도 포함되어 있는 상태
        ai_message = ai_response.choices[0].message

        # 대화내역에 저장
        messages.append(ai_message)

        # 응답 텍스트만 출력
        print("AI\t", ai_message.content)
        print()

대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.


사용자	: 안녕?


ChatCompletionMessage(content='안녕하세요! 무엇을 도와드릴까요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


사용자	: 지금 몇시야?


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_GqwsAP0mUThCQ6v6bEXiafFZ', function=Function(arguments='{}', name='get_current_time'), type='function')])
AI	 안녕하세요! 지금 시간은 2026년 4월 1일 오후 3시 0분입니다. 무엇을 도와드릴까요?



사용자	: 워싱턴은 몇시야?


ChatCompletionMessage(content='워싱턴 D.C.는 미국 동부 표준시(EST) 또는 동부 일광 절약 시간제(EDT)를 따릅니다. 현재 일광 절약 시간제 기간이며, 한국보다 13시간이 늦습니다. 그러므로 지금 워싱턴 D.C.의 시간은 2026년 4월 1일 오전 2시 0분입니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


사용자	: 프랑스는 몇시야?


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_mGWD4hWR2bAOQlndsPL6uQlL', function=Function(arguments='{}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_YmDm6ll5oRuQB4sPdw4dXIgp', function=Function(arguments='{}', name='get_current_time'), type='function')])


BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_YmDm6ll5oRuQB4sPdw4dXIgp", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

## **3. 사용자의 현재 위치에 따라, 해외 시간을 정확히 판단하는 모델**
- 한번에 여러 도구를 호출가능 하도록 `병렬 호출 기능`도 적용

In [13]:
# pytz: 세계 모든 국가의 시간 계산을 지원하는 모듈
import pytz

In [12]:
# 예시
# timezone: 문자열로 된 대륙/도시 이름을 파이썬이 이해하는 시간 객체로 변경하는 함수
pytz.timezone("Asia/Seoul")   # (대륙/도시)

<DstTzInfo 'Asia/Seoul' LMT+8:28:00 STD>

In [15]:
temp = pytz.timezone("Asia/Seoul")
# localize: 세계 표준시를 반영하여 현재 지역 시간으로 변환
print(temp.localize(datetime.now()))

2026-04-01 15:32:19.344119+09:00


In [13]:
# 사용가능한 대륙 및 도시 리스트
pytz.common_timezones

['Africa/Abidjan',
 'Africa/Accra',
 'Africa/Addis_Ababa',
 'Africa/Algiers',
 'Africa/Asmara',
 'Africa/Bamako',
 'Africa/Bangui',
 'Africa/Banjul',
 'Africa/Bissau',
 'Africa/Blantyre',
 'Africa/Brazzaville',
 'Africa/Bujumbura',
 'Africa/Cairo',
 'Africa/Casablanca',
 'Africa/Ceuta',
 'Africa/Conakry',
 'Africa/Dakar',
 'Africa/Dar_es_Salaam',
 'Africa/Djibouti',
 'Africa/Douala',
 'Africa/El_Aaiun',
 'Africa/Freetown',
 'Africa/Gaborone',
 'Africa/Harare',
 'Africa/Johannesburg',
 'Africa/Juba',
 'Africa/Kampala',
 'Africa/Khartoum',
 'Africa/Kigali',
 'Africa/Kinshasa',
 'Africa/Lagos',
 'Africa/Libreville',
 'Africa/Lome',
 'Africa/Luanda',
 'Africa/Lubumbashi',
 'Africa/Lusaka',
 'Africa/Malabo',
 'Africa/Maputo',
 'Africa/Maseru',
 'Africa/Mbabane',
 'Africa/Mogadishu',
 'Africa/Monrovia',
 'Africa/Nairobi',
 'Africa/Ndjamena',
 'Africa/Niamey',
 'Africa/Nouakchott',
 'Africa/Ouagadougou',
 'Africa/Porto-Novo',
 'Africa/Sao_Tome',
 'Africa/Tripoli',
 'Africa/Tunis',
 'Africa/Wi

In [6]:
# 세계 대륙 및 도시에 따른 시간 출력 함수 (디폴트는 서울로 설정)
def get_current_time_2(timezone: str="Asia/Seoul") :    # 매개변수: 타입 힌트 ="기본값"
    try :
        # 타임존 객체 생성 (대륙/도시 문자열이 잘못 들어오면 에러가 발생)
        tz = pytz.timezone(timezone)

        # 세계 타임존을 반영한 현재 시간
        now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")

        return f"{now} ({timezone})"  # 문자열로 반환되게 설정!!!!

    # 모델이 잘못된 타임존을 보냈을 때, 프로그램이 중단되지 않고 에러 메시지를 반환하도록 처리
    # (타임존 형식을 못 맞추는 경우가 있을 수 있으므로 에러를 반환해서 해결하게 유도)
    except pytz.UnknownTimeZoneError :
        return f"""Error: '{timezone}'은(는) 유효하지 않은 타임존입니다.
        'Asia/Seoul', 'America/New_York', 'Europe/Paris'와 같은 형식으로 다시 요청하세요.
        잘 모르겠으면 pytz.common_timezones 코드를 실행해서 결과를 확인해주세요.
        """

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time_2",
            "description": "해당 지역과 도시의 날짜와 시간을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    # 도구(함수)에 입력되는 파라미터(매개변수) 명
                    "timezone": {
                        # <type 종류>
                        # string(문자열), integer(정수), number(정수,실수), boolean(논리)
                        "type": "string",
                        "description": "현재 날짜와 시간을 반환할 지역과 도시를 입력하세요.(예: Asia/Seoul)"
                    }                    
                },
                # 도구(함수) 구동에 반드시 필요한 파라미터(매개변수) 지정
                "required": ["timezone"]
            }
        }
    }
]

In [ ]:
# 초기 시스템 프롬프트 설정
messages = [
    {"role": "system", "content": "너는 사용자의 질문에 맞게 친절히 응답하는 응대원이야."}
]

print("대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.")

while True :
    user_input = input("사용자\t:")

    if user_input == "종료" :
        print("=== 대화를 종료 합니다 ===")
        break

    messages.append({"role":"user", "content":user_input})

    ai_response = get_ai_response(messages, tools=tools)

    #print(ai_response.choices[0].message)

    ai_message = ai_response.choices[0].message

    tool_calls = ai_message.tool_calls

    if tool_calls :
        messages.append(ai_message)

        # ====================== 변경 부분 ======================
        # AI가 여러개의 도구를 호출하면 tool_calls는 리스트로 반환되므로 반복문 설정
        for tool in tool_calls :
            tool_name = tool.function.name   # 이름과 주문번호 확인!!!
            tool_call_id = tool.id

            # tool의 매개변수(arguments)은 JSON 형태의 문자열이므로 파싱해서 딕셔너리로 활용
            tool_args = json.loads(tool.function.arguments)

            if tool_name == "get_current_time_2" :
                # 모델이 추출한 타임존 정보를 함수에 전달
                timezone_arg = tool_args.get("timezone")

                ####### 확인용 코드
                print(f" ### 확인용 -> 함수 실행: {tool_name}, 매개변수: {timezone_arg}")

                # 모델이 만든 매개변수 명을 도구에 입력
                function_result = get_current_time_2(timezone_arg)

                # 도구 실행 정보를 대화내역에 추가
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    # content는 반드시 문자열이어야 하므로 안전하게 한번 더 형변환
                    "content": str(function_result)
                })

        # ======================================================

        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print("AI\t", ai_message.content)
    print()

대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.


사용자	: 미국 시간 몇시야?


 ### 확인용 -> 함수 실행: get_current_time_2, 매개변수: America/New_York
 ### 확인용 -> 함수 실행: get_current_time_2, 매개변수: America/Chicago
 ### 확인용 -> 함수 실행: get_current_time_2, 매개변수: America/Denver
 ### 확인용 -> 함수 실행: get_current_time_2, 매개변수: America/Los_Angeles
AI	 현재 미국의 주요 도시 시간은 다음과 같습니다:

- 뉴욕 (동부 시간): 2026년 4월 1일, 오전 3시 14분
- 시카고 (중부 시간): 2026년 4월 1일, 오전 2시 14분
- 덴버 (산악 시간): 2026년 4월 1일, 오전 1시 14분
- 로스앤젤레스 (태평양 시간): 2026년 4월 1일, 자정 14분

다른 지역의 시간이 필요하시면 말씀해 주세요!



사용자	: 종료


=== 대화를 종료 합니다 ===


## 4. 주식 정보 질문에 응답하는 모델 구동

In [7]:
import FinanceDataReader as fdr

In [ ]:
# 주가 정보 조회 도구(함수) 선언
def get_stock_price(ticker_symbol: str) :
    "FinanceDataReader를 사용하여 해당 종목의 현재가와 등락률을 반환합니다."
    try :
        # 토큰의 길이가 너무 커지지 않게 최근 1년 데이터로 최대 기간 고정
        date_start = datetime.today().date() - timedelta(days=365)

        # 데이터 조회 (종목코드, 시작일)
        df = fdr.DataReader(ticker_symbol, date_start)

        # 데이터프레임이 비었다면(행이 0개라면) True 반환
        if df.empty :
            return f"Error: '{ticker_symbol}'에 대한 데이터가 없습니다. 종목 코드를 확인해주세요."

        # 데이터프레임 전체를 CSV 문자열로 변환
         # to_csv: CSV형태로 변환(쉼표(,)로 구분된 텍스트)
         # 반환되는 값 중에 index가 날짜정보이기 때문에 index=True로 같이 포함 시켜줌
        csv_string = df.to_csv(index=True)

        # 모델에게 문자열로 데이터 정보를 반환함
        return f"종목코드: {ticker_symbol}의 전체 주가 데이터(CSV형식):\n{csv_string}"

    except Exception as e :
        return f"에러 발생: {e}  -> 에러를 확인하고 조치하세요."




# 도구 명세서 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "특정 주식 종목의 전체 주가 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker_symbol": {
                        "type": "string",
                        "description": """조회할 주식의 종목 코드 혹은 티커 심볼
                        (예: 삼성전자는 005930, 애플은 AAPL, 비트코인은 BTC/KRW, 환율은 USD/KRW)
                        """
                    }                    
                },
                "required": ["ticker_symbol"]
            }
        }
    }
]

In [ ]:
# 모델 구동 코드
# 초기 시스템 프롬프트 설정
messages = [
    {"role": "system",
    "content": """너는 주식 전문가야. 한국 주식은 6자리 숫자 코드(예: 005930)로, 미국 주식은 티커(예: AAPL)로,
    암호화폐는 KRW 마켓 기준(예: BTC/KRW)로 도구를 사용해."""}
]

print("대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.")

while True :
    user_input = input("사용자\t:")

    if user_input == "종료" :
        print("=== 대화를 종료 합니다 ===")
        break

    messages.append({"role":"user", "content":user_input})

    # 1차 응답 요청
    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    tool_calls = ai_message.tool_calls      # 도구 사용 정보 추출
    
    # AI가 도구 사용을 판단한 경우~~~
    if tool_calls :
        # 도구 정보가 포함된 메시지 객체를 append!
        messages.append(ai_message)

        print(f"\n[System] AI가 {len(tool_calls)}개의 도구 호출을 요청했습니당.")

        for tool in tool_calls :
            tool_name = tool.function.name          # 도구명
            tool_call_id = tool.id                  # 도구 id
            tool_args = json.loads(tool.function.arguments)
            # tool.function.arguments: LLM이 파이썬 코드에 넘겨주기 위해 최종적으로 뱉어내는 결과물. 도구에 입력한 매개변수.
            # 이게 dict처럼 보여도 str임. 그래서 json.loads 써서 dict로 반환해주는 것~~~

            if tool_name == 'get_stock_price' :
                ticker_arg = tool_args.get('ticker_symbol')
                print(f" ### 도구 실행 -> 종목 : {ticker_arg}")

                # 도구 실행
                function_result = get_stock_price(ticker_symbol=ticker_arg)

                # 도구 실행 내역을 대화 내역에 추가!!
                # OpenAI가 정해둔 양식에 맞춰서 줘야, LLM이 에러없이 다음 대답을 만들어낼 수 있음
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call_id,    # id가 있으니까 병렬호출 가능하게 함
                    'name': tool_name,
                    'content': str(function_result)
                })

        # 도구 실행 결과가 포함된 대화내역으로 최종 답변 요청
        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    # 대화 내역에 저장 후 최종 답변 출력
    messages.append(ai_message)
    print("AI\t", ai_message.content)
    print()



대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.

[System] AI가 1개의 도구 호출을 요청했습니당.
 ### 도구 실행 -> 종목 : 005930
AI	 삼성전자(종목코드: 005930)의 최근 일주일 간의 주가 변동 추이는 다음과 같습니다.

- **2026-03-26**: 시가 185,500원, 고가 185,900원, 저가 178,900원, 종가 180,100원 (-4.71%)
- **2026-03-27**: 시가 172,100원, 고가 181,700원, 저가 172,000원, 종가 179,700원 (-0.22%)
- **2026-03-30**: 시가 171,000원, 고가 176,650원, 저가 170,600원, 종가 176,300원 (-1.89%)
- **2026-03-31**: 시가 170,000원, 고가 174,700원, 저가 167,000원, 종가 167,200원 (-5.16%)
- **2026-04-01**: 시가 179,000원, 고가 190,800원, 저가 178,000원, 종가 189,600원 (+13.40%)
- **2026-04-02**: 시가 192,600원, 고가 193,600원, 저가 178,800원, 종가 180,150원 (-4.98%)

이 기간 동안 주가는 다양한 변동을 보이며, 중간에 하락 후 반등하는 모습을 보였습니다.


[System] AI가 2개의 도구 호출을 요청했습니당.
 ### 도구 실행 -> 종목 : 005930
 ### 도구 실행 -> 종목 : 000660
AI	 삼성전자(종목코드: 005930)와 SK하이닉스(종목코드: 000660)의 최근 3일 간 주가 변동 추이는 다음과 같습니다.

### 삼성전자 (005930)
- **2026-03-31**: 시가 170,000원, 고가 174,700원, 저가 167,000원, 종가 167,200원 (-5.16%)
- **2026-04-01**: 시가 179,000원, 고가 190,800원, 저가 178,000원, 종가 189,600원 (+13.40%

## **5. 현재시간, 세계시간, 주식정보 세가지 Tool 사용하는 모델 구현**

In [ ]:
# 함수 선언 -> 기존에 선언되어 있는 도구들 활용
 # get_current_time(), get_current_time_2(), get_stock_price()

In [ ]:
### 도구 명세서 설정 (3가지 tool에 대한 정보가 들어가야 함)

tools = [
    # ----------- 첫번째 도구 --------------
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS'형태의 문자열(string)로 반환합니다.",
            "parameters": {
                "type": "object",  # type은 object(딕셔너리 형태)로 고정
                "properties": {}   # 파라미터명 (없으면 비워두기)
            }
        }
    },  # <-- 콤마
    # ----------- 두번째 도구 --------------
    {
        "type": "function",
        "function": {
            "name": "get_current_time_2",
            "description": "해당 지역과 도시의 날짜와 시간을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": { 
                    # (함수)에 입력되는 파라미터(매개변수) 명
                    "timezone": {
                        "type": "string",
                        "description": "현재 날짜와 시간을 반환할 지역과 도시를 입력하세요.(예: Asia/Seoul)"
                    }                    
                },
                # 도구(함수) 구동에 반드시 필요한 파라미터(매개변수) 지정
                "required": ["timezone"]
            }
        }
    },
    # ----------- 세번째 도구 --------------
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "특정 주식 종목의 전체 주가 정보를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker_symbol": {
                        "type": "string",
                        "description": """조회할 주식의 종목 코드 혹은 티커 심볼
                        (예: 삼성전자는 005930, 애플은 AAPL, 비트코인은 BTC/KRW, 환율은 USD/KRW)"""
                    }                    
                },
                "required": ["ticker_symbol"]
            }
        }
    }
]

In [ ]:
### 모델 구동

# 함수 객체를 찾기 위한 매핑 딕셔너리 작성
# (도구 개수가 늘어날 때마다 조건문을 사용하는 것은 비효율적)
# 현업에서는 보통 5개정도 사용함!!
available_tools = {
    # key에는 명칭을 문자열로, value에는 객체 자체를 입력
    "get_current_time": get_current_time,
    "get_current_time_2": get_current_time_2,
    "get_stock_price": get_stock_price
}

messages = [
    {"role": "system",
    "content": """너는 주식 및 시간 정보 전문가야.
    1. 한국 주식은 6자리 코드(예: 005930), 미국 주식은 티커(예: AAPL), 암호화폐는 KRW 마켓기준(예: BTC/KRW)으로 도구(get_stock_price)를 사용해.
    2. 주식 가격을 조회할 때는 시장 개장시간이나 현재시간을 확인하지 말고, 즉시 주식 가격 조회 도구(get_stock_price)만 단독 사용해.
    3. 사용자가 그냥 시간을 물으면 기본 시간 함수(get_current_time)를 사용하고,
    특정 지역(도시)의 시간을 물으면 타임존 함수(get_current_time_2)를 사용해."""}
]

print("대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.")

while True :
    user_input = input("사용자\t:")
    print('User\t', user_input)

    if user_input == "종료" :
        print("=== 대화를 종료 합니다 ===")
        break

    messages.append({"role":"user", "content":user_input})

    # 1차 응답 요청
    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    tool_calls = ai_message.tool_calls 

    if tool_calls :
        messages.append(ai_message)

        print(f"\n[System] AI가 {len(tool_calls)}개의 도구 호출을 요청했습니당.")

        for tool in tool_calls :
            tool_name = tool.function.name          # 도구명
            tool_call_id = tool.id                  # 도구 id
            tool_args = json.loads(tool.function.arguments)


            # ------------- 위에 선언해둔 도구 매핑 딕셔너리 활용 ------------
            tool_to_run = available_tools.get(tool_name)

            if tool_to_run :
                print(f" ▶ 실행 중: {tool_name} | 매개변수: {tool_args}")

                # **로 딕셔너리 형태인 tool_args의 key에 따른 value를 매개변수로 입력
                # 매개변수가 없으면 빈 괄호 (), 있으면 (ticker_symbol="~~") 형태로 변환)
                func_result = tool_to_run(**tool_args)   # ** 딕셔너리 언패킹!!!

                # 도구 사용 결과 저장
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    "content": str(func_result)
                })

            else:
                print(f'Error: {tool_name} 도구를 찾을 수 없습니다. 다시 판단해 선택해주세요.')


        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)
    print("AI\t", ai_message.content)
    print()


대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.
User 워싱턴 지금 몇시야?

[System] AI가 1개의 도구 호출을 요청했습니당.
 ▶ 실행 중: get_current_time_2 | 매개변수: {'timezone': 'America/New_York'}
AI	 워싱턴의 현재 시간은 2026년 4월 1일 23시 2분입니다.

User 
AI	 무엇을 도와드릴까요?

User 최근 3일간 삼전, sk하닉 주식 가격 어떻게 변했어?

[System] AI가 2개의 도구 호출을 요청했습니당.
 ▶ 실행 중: get_stock_price | 매개변수: {'ticker_symbol': '005930'}
 ▶ 실행 중: get_stock_price | 매개변수: {'ticker_symbol': '000660'}
AI	 최근 3일 동안의 삼성전자와 SK하이닉스 주식 가격 변동은 다음과 같습니다:

### 삼성전자 (005930)
- **2026년 3월 31일:** 시가 170,000원, 고가 174,700원, 저가 167,000원, 종가 167,200원
- **2026년 4월 1일:** 시가 179,000원, 고가 190,800원, 저가 178,000원, 종가 189,600원
- **2026년 4월 2일:** 시가 192,600원, 고가 193,600원, 저가 177,300원, 종가 178,900원

### SK하이닉스 (000660)
- **2026년 3월 31일:** 시가 817,000원, 고가 852,000원, 저가 806,000원, 종가 807,000원
- **2026년 4월 1일:** 시가 884,000원, 고가 905,000원, 저가 858,000원, 종가 893,000원
- **2026년 4월 2일:** 시가 912,000원, 고가 914,000원, 저가 838,000원, 종가 844,000원

삼성전자와 SK하이닉스 모두 최근 3일 동안 가격 변동폭이 있었습니다.

User 종료
=== 대화를 종료 합니다 ===


---
## **[실습] 나만의 실시간 날씨&뉴스 AI 비서 만들기**
- **1. Tool 만들기**
    - (1) 특정 지역을 입력받아 네이버 검색 결과에서 실시간 온도와 날씨 상태를 크롤링(BeautifulSoup활용)하는 `get_weather_info(location)` 도구를 완성하세요.
        - ex) location에 '서울'을 입력하면 서울의 날씨가 검색되어야 함
        - try-except문을 활용하여 크롤링시 발생할 수 있는 예외상황에서도 코드가 멈추지 않도록 하세요.
    - (2) 특정 키워드를 입력받아 네이버 뉴스 검색 결과에서 최신 뉴스 헤드라인 3개를 크롤링하는 `search_news(keyword)` 도구를 완성하세요.

- **2. 도구 명세서(JSON) 작성**
    - AI가 이 두 가지 함수를 이해할 수 있도록 tools 리스트 안에 JSON 형태로 정의하세요.

- **3. 연동 로직 채우기**
    - 'if tool_calls :' 내부의 로직을 완성하여, AI가 도구를 병렬로 호출했을 때 결과를 다시 메시지에 담아 최종 답변을 받도록 만드세요.



※ AI가 멀티턴 대화를 할 수 있도록 코드를 구성하세요.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import requests
from bs4 import BeautifulSoup

# API 키 로드
load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=MY_API_KEY)

# AI 응답 함수
def get_ai_response(messages, tools=None) :
    completion = client.chat.completions.create(
        model="gpt-4o",
        # [TODO 1] LLM에게 대화 기록(messages)과 사용 가능한 도구(tools)를 전달하세요.
        messages=messages,
        tools=available_tools
    )
    return completion

# 날씨 정보 함수
def get_weather_info(location:str) :
    url = f'https://search.naver.com/search.naver?&query={location}+날씨'
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

    try :
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        temp = soup.select_one('.temperature_text')
        weather = soup.select_one('.weather.before_slash')

        # 데이터가 제대로 찾아졌는지 확인
        if temp and weather:
            # "현재 온도23°" 처럼 나오므로, 불필요한 글자를 날려주면 더 깔끔합니다 (이건 선택사항)
            current_degrees = temp.text.replace('현재 온도', '').strip()
            current_weather = weather.text.strip()
            
            return f'{location}의 기온은 {current_degrees}, 날씨는 {current_weather}입니다.'
    
    except Exception as e:
        return f'정보를 불러오는데 실패했습니다. {e}'
    

# 뉴스 함수
def search_news(keyword:str) :
    url = f'https://search.naver.com/search.naver?sm=tab_hty.top&where=news&ssc=tab.news.all&query={keyword}&sm=tab_opt&sort=1'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

    try :
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        news_tags = soup.select('span.sds-comps-text-ellipsis-1.sds-comps-text-type-headline1')

        # 상위 3개만 가져오기 (리스트 슬라이싱 [:3] 활용)
        top_3_news = news_tags[:3]
        
        # 결과물로 만들 문자열 조합하기
        result_text = f"[{keyword} 최신 뉴스 헤드라인 Top 3]\n"
        for i, tag in enumerate(top_3_news, start=1):
            title = tag.text
            result_text += f"{i}. {title}\n"
            
        return result_text.strip()

    except Exception as e:
        return f'정보를 불러오는데 실패했습니다. {e}'
    


### =============== 도구 명세서 설정

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_info",
            "description": "해당 지역의 기온과 날씨를 문자열로 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        'type': 'string',
                        'description': '날씨를 검색할 지역 이름을 입력하세요. (예: 서울, 제주도, 부산)'
                    }
                },
                "required": ['location']   # 무조건 지역 이름을 뽑아내도록 강제!!!!
            }
        }
    }, 
    {
        "type": "function",
        "function": {
            "name": "search_news",
            "description": "해당 키워드의 뉴스 타이틀 3가지를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": { 
                    "keyword": {
                        "type": "string",
                        "description": "검색할 뉴스 키워드를 입력하세요. (예: 인공지능, 주식, 벚꽃)"
                    }                    
                },
                "required": ["keyword"]
            }
        }
    }
]


### ================= 모델 구동

available_tools = {
    "search_news": search_news,
    "get_weather_info": get_weather_info,
}

messages = [
    {"role": "system",
    "content": """너는 날씨 및 뉴스 기사 분석 전문가야.
    1. 사용자가 특정 지역의 날씨나 기온을 물어보면, `get_weather_info` 도구를 사용해서 정확한 정보를 제공해.
    2. 사용자가 특정 키워드의 소식이나 뉴스를 물어보면, 반드시 `search_news` 도구를 사용해.
    3. 도구가 반환한 결과(데이터)를 그대로 앵무새처럼 읊지 말고, 네가 직접 읽고 자연스럽고 친절한 대화체로 요약해서 전달해줘.
    4. 만약 두 가지 도구를 다 써야 하는 상황(예: "오늘 서울 날씨랑 빅데이터분석기사 관련 뉴스 좀 알려줘")이라면, 두 도구를 모두 사용한 뒤에 종합해서 브리핑해줘.
    5. 검색에 실패하거나 에러가 났다는 결과가 오면, 사용자에게 정중하게 사과하고 검색어를 바꿔서 다시 질문해 달라고 안내해."""}
]

print("대화를 시작합니다. 종료하시려면 '종료'를 입력해주세요.")

while True :
    user_input = input("사용자\t:")
    print('User\t', user_input)

    if user_input == "종료" :
        print("=== 대화를 종료 합니다 ===")
        break

    messages.append({"role":"user", "content":user_input})

    # 1차 응답 요청
    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    tool_calls = ai_message.tool_calls 

    # 도구 사용이 필요하다면, 도구를 쓰겠다고 한 메시지 전체를 messages에 기록하기
    if tool_calls :
        messages.append(ai_message)

        print(f"\n[System] AI가 {len(tool_calls)}개의 도구 호출을 요청 ⌛")

        for tool in tool_calls :
            tool_name = tool.function.name          # 도구명
            tool_call_id = tool.id                  # 도구 id
            tool_args = json.loads(tool.function.arguments)


            # ------------- 위에 선언해둔 도구 매핑 딕셔너리 활용 ------------
            tool_to_run = available_tools.get(tool_name)

            if tool_to_run :
                print(f" ▶ 실행 중: {tool_name} | 매개변수: {tool_args}")

                func_result = tool_to_run(**tool_args)   # ** 딕셔너리 언패킹!!!

                # 도구 사용 결과 저장
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    "content": str(func_result)
                })

            else:
                print(f'Error: {tool_name} 도구를 찾을 수 없습니다. 다시 판단해 선택해주세요.')

        # 도구 실행 결과가 포함된 messages를 다시 LLM에게 보내 최종 답변을 받으세요.
        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    # 최종 답변 저장 및 출력
    messages.append(ai_message)
    print("AI\t", ai_message.content)
    print()


In [74]:
get_weather_info('서울')

'서울의 기온은 17.5°, 날씨는 맑음입니다.'

In [54]:
search_news('고양이')

'[고양이 최신 뉴스 헤드라인 Top 3]\n1. 주말에 떠나기 좋은 공주 가볼 만한 곳 BEST 코스\n2. 고양시, 공연 넘어 소비 창출… 체류형 관광 ‘고양콘트립’ 가동\n3. 반려동물 행동교정 무료 지원 확대”… 영등포구,‘우리집 댕냥이 행동...'

---
### 뉴스 불러올 때 api 활용하는 방법도 있음

In [ ]:
# 뉴스 함수 api 활용

def search_news_api(keyword:str) :

    load_dotenv()
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")

    url = f'https://openapi.naver.com/v1/search/news.json'

    # 3. 출입증을 헤더(headers)에 담아서 보여줍니다.
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    
    # 4. 검색 옵션 세팅 (핵심!)
    params = {
        "query": keyword, # 검색어
        "display": 3,     # 몇 개
        "sort": "date"    # date: 최신순, sim: 관련도순 (기본값)
    }
    
    try:
        # 서버에 요청 보내기 (이번엔 url, headers, params를 싹 다 넣어서 보냅니다)
        response = requests.get(url, headers=headers, params=params)
        
        # 받아온 JSON 데이터를 파이썬 딕셔너리로 변환
        data = response.json()
        
        # 검색 결과가 'items'라는 리스트 안에 예쁘게 들어있습니다.
        result_text = f"[{keyword} 최신 뉴스 Top 3 (API)]\n"
        
        for i, item in enumerate(data['items'], start=1):
            # API는 검색어를 강조하기 위해 제목에 <b> 태그를 넣어서 주므로 제거해줍니다.
            title = item['title'].replace('<b>', '').replace('</b>', '').replace('&quot;', '"')
            # 2. 요약(description) 태그 클렌징
            description = item['description'].replace('<b>', '').replace('</b>', '').replace('&quot;', '"')

            result_text += f"{i}. {title}\n"
            result_text += f"   요약: {description[:100]}...\n\n"
            
        return result_text.strip()
        
    except Exception as e:
        return f"뉴스 API 호출에 실패했습니다. 에러: {e}"

In [72]:
search_news_api('고양이')

'[고양이 최신 뉴스 Top 3 (API)]\n1. 영등포구, \'2026년 우리집 댕냥이 행동개선 프로젝트\'\n   요약: 특히, 그동안 소외되었던 반려묘 가정까지 포함해 고양이 집사들도 맞춤형 교육을 받을 수 있도록 폭넓은 기회를 마련했다. 프로그램은 전문 훈련사가 사전 전화상담을 통해 반려동물의 문...\n\n2. 한 번 사면 평생 입어요, 윤승아가 요즘 매일 입는 기본템은 ‘이것’\n   요약: 고양이 자수가 들어간 보드(BODE) 티셔츠 안에 입은 얇은 긴 팔 티셔츠 역시 오라리 제품인데요. 살이 많이 비쳐서 주로 이너로 입는데요. 텐션이 좋아 팔이 얇아 보이는 효과도 ...\n\n3. 에스앤씨컴퍼니, 라온메딕스 방어복 리뉴얼…"피폭 위험 줄여"\n   요약: 특히 강아지, 고양이의 특성상 의료진이 직접 고정하거나 가까운 거리에서 촬영을 보조하는 경우가 많아 반복적이고 근접한 방사선 노출 환경에 놓였다는 지적이 제기된다. 이에 에스앤씨컴...'